# Data Generation and Upload to watsonx.ai

This notebook generates synthetic input data for the TensorFlow model and uploads it as a CSV file to watsonx.ai project assets.

## Workflow
1. Setup and Configuration
2. Generate Synthetic Input Data
3. Save Data as CSV
4. Upload CSV to watsonx.ai Project Assets
5. Verify Upload

## 1. Setup and Configuration

In [ ]:
# Import required libraries
import os
import json
import numpy as np
import pandas as pd
from datetime import datetime, timezone, timedelta
from importlib.metadata import version
import warnings
warnings.filterwarnings('ignore')

# Set timezone to UTC+8 (Asia/Taipei)
TZ_UTC8 = timezone(timedelta(hours=8))

# Get package versions
np_version = version('numpy')
pd_version = version('pandas')
watsonx_version = version('ibm-watsonx-ai')

print("="*80)
print("PACKAGE VERSIONS")
print("="*80)
print(f"✅ NumPy version: {np_version}")
print(f"✅ Pandas version: {pd_version}")
print(f"✅ IBM watsonx.ai version: {watsonx_version}")
print("\n✅ All libraries imported successfully")

### Load Environment Variables

This cell loads the required credentials from environment variables. These should be set in your watsonx.ai project environment.

In [ ]:
# Load credentials from environment variables
PROJECT_ID = os.getenv('PROJECT_ID')
SPACE_ID = os.getenv('SPACE_ID')
WX_API_KEY = os.getenv('WX_API_KEY')

print("="*80)
print("ENVIRONMENT VARIABLES CHECK")
print("="*80)
print(f"PROJECT_ID: {'✅ Set' if PROJECT_ID else '❌ Not set'}")
print(f"SPACE_ID: {'✅ Set' if SPACE_ID else '❌ Not set'}")
print(f"WX_API_KEY: {'✅ Set' if WX_API_KEY else '❌ Not set'}")
print("="*80)

if not all([PROJECT_ID, WX_API_KEY]):
    print("\n⚠️ WARNING: Required environment variables are missing!")
    print("Please set the following environment variables:")
    if not PROJECT_ID:
        print("  - PROJECT_ID")
    if not WX_API_KEY:
        print("  - WX_API_KEY")
    print("\nYou can set them in the project settings or manually in the code.")
else:
    print("\n✅ All required environment variables are set!")

## 2. Define Input Configuration

Based on the model architecture, we define the configuration for all 32 inputs.

In [ ]:
# Input configuration from the model
inputs_config = [
    # BLOCK_MD_BRN_CONTACTS
    ("input_x_o_MD_BRN_CONTACTS", (16,), "float"),
    ("input_x_c_1_MD_BRN_CONTACTS", (1,), "int", 12),
    ("input_x_c_2_MD_BRN_CONTACTS", (1,), "int", 6),
    ("input_x_c_3_MD_BRN_CONTACTS", (1,), "int", 3),
    ("input_x_c_4_MD_BRN_CONTACTS", (1,), "int", 6),
    ("input_x_c_5_MD_BRN_CONTACTS", (1,), "int", 7),
    ("input_x_c_6_MD_BRN_CONTACTS", (1,), "int", 10),
    ("input_x_c_7_MD_BRN_CONTACTS", (1,), "int", 104),
    ("input_x_c_8_MD_BRN_CONTACTS", (1,), "int", 8),
    
    # Time series
    ("input_x_t_MD_CASH_FLOW", (12, 22), "float"),
    ("input_x_t_MD_CUS_EXCH", (12, 12), "float"),
    ("input_x_t_MD_CUS_LOAN", (12, 24), "float"),
    
    # BLOCK_MD_CUS_RISK
    ("input_x_o_MD_CUS_RISK", (1,), "float"),
    ("input_x_c_1_MD_CUS_RISK", (1,), "int", 7),
    ("input_x_c_2_MD_CUS_RISK", (1,), "int", 6),
    ("input_x_c_3_MD_CUS_RISK", (1,), "int", 5),
    ("input_x_c_4_MD_CUS_RISK", (1,), "int", 6),
    ("input_x_c_5_MD_CUS_RISK", (1,), "int", 6),
    ("input_x_c_6_MD_CUS_RISK", (1,), "int", 5),
    ("input_x_c_7_MD_CUS_RISK", (1,), "int", 7),
    ("input_x_c_8_MD_CUS_RISK", (1,), "int", 5),
    
    # BLOCK_MD_CUS_STYLE
    ("input_x_o_MD_CUS_STYLE", (35,), "float"),
    ("input_x_c_1_MD_CUS_STYLE", (1,), "int", 3),
    ("input_x_c_2_MD_CUS_STYLE", (1,), "int", 5),
    ("input_x_c_3_MD_CUS_STYLE", (1,), "int", 113),
    ("input_x_c_4_MD_CUS_STYLE", (1,), "int", 112),
    ("input_x_c_5_MD_CUS_STYLE", (1,), "int", 7),
    ("input_x_c_6_MD_CUS_STYLE", (1,), "int", 23),
    ("input_x_c_7_MD_CUS_STYLE", (1,), "int", 171),
    
    # More time series
    ("input_x_t_MD_DEP_AUM", (12, 12), "float"),
    ("input_x_t_MD_NEW_INVST", (12, 21), "float"),
    ("input_x_t_MD_PDH_AUM", (12, 30), "float"),
]

print(f"✅ Configured {len(inputs_config)} input features")

## 3. Generate Synthetic Data

Generate synthetic data samples for model testing.

In [ ]:
def generate_single_sample(seed=None):
    """Generate a single sample of data"""
    if seed is not None:
        np.random.seed(seed)
    
    sample_data = {}
    
    for config in inputs_config:
        name = config[0]
        shape = config[1]
        dtype = config[2]
        
        if dtype == "int":
            vocab_size = config[3]
            # Generate valid integer indices [0, vocab_size)
            data = np.random.randint(0, vocab_size, size=shape)
        else:
            # Generate float data
            data = np.random.randn(*shape)
        
        # Flatten multi-dimensional arrays for CSV storage
        if len(shape) > 1:
            # For time series data, flatten to 1D
            sample_data[name] = data.flatten().tolist()
        else:
            sample_data[name] = data.tolist()
    
    return sample_data


# Generate multiple samples
num_samples = 100  # Generate 100 samples
print(f"\n{'='*80}")
print(f"Generating {num_samples} synthetic data samples...")
print(f"{'='*80}")

samples = []
for i in range(num_samples):
    sample = generate_single_sample(seed=42 + i)
    samples.append(sample)

print(f"✅ Generated {len(samples)} samples")
print(f"✅ Each sample has {len(samples[0])} features")

## 4. Convert to DataFrame and Save as CSV

In [ ]:
# Convert to DataFrame
df = pd.DataFrame(samples)

print(f"\n{'='*80}")
print("DataFrame Information")
print(f"{'='*80}")
print(f"Shape: {df.shape}")
print(f"Columns: {len(df.columns)}")
print(f"\nFirst few rows:")
print(df.head())

# Generate timestamp for file naming
timestamp = datetime.now(TZ_UTC8).strftime("%Y%m%d_%H%M%S")
csv_filename = f"model_input_data_{timestamp}.csv"

# Save to CSV
df.to_csv(csv_filename, index=False)

print(f"\n✅ Data saved to: {csv_filename}")
print(f"✅ File size: {os.path.getsize(csv_filename) / 1024:.2f} KB")

## 5. Connect to watsonx.ai

**Important**: You need to provide your watsonx.ai credentials.

In [ ]:
from ibm_watsonx_ai import APIClient
from ibm_watsonx_ai import Credentials

# Use credentials from environment variables
credentials = Credentials(
    url="https://us-south.ml.cloud.ibm.com",  # Replace with your region
    api_key=WX_API_KEY
)

client = APIClient(credentials)

print("="*80)
print("watsonx.ai Connection")
print("="*80)
print(f"✅ Connected to: {credentials.url}")
print(f"✅ Client version: {client.version}")

## 6. Set Project Context

In [ ]:
# Use PROJECT_ID from environment variables
client.set.default_project(PROJECT_ID)

print(f"✅ Default project set to: {PROJECT_ID}")

## 7. Upload CSV to Project Assets

In [ ]:
print(f"\n{'='*80}")
print("Uploading CSV to watsonx.ai Project Assets")
print(f"{'='*80}")

# Upload the CSV file as a data asset
asset_details = client.data_assets.create(
    name=f"training_data_{timestamp}",
    file_path=csv_filename
)

training_data_asset_id = client.data_assets.get_id(asset_details)

print(f"✅ CSV uploaded successfully")
print(f"✅ Asset ID: {training_data_asset_id}")
print(f"✅ Asset Name: {asset_details['metadata']['name']}")

## 8. Verify Upload

In [ ]:
# List all data assets in the project
print(f"\n{'='*80}")
print("Verifying Upload - Listing Project Data Assets")
print(f"{'='*80}")

assets = client.data_assets.get_details()

print(f"\nTotal data assets in project: {len(assets['resources'])}")
print(f"\nRecent uploads:")
for asset in assets['resources'][:5]:  # Show first 5
    print(f"  - {asset['metadata']['name']} (ID: {asset['metadata']['asset_id']})")

print(f"\n{'='*80}")
print("✅ Data Generation and Upload Complete!")
print(f"{'='*80}")
print(f"\n📊 Summary:")
print(f"  - Generated samples: {num_samples}")
print(f"  - Features per sample: {len(inputs_config)}")
print(f"  - CSV file: {csv_filename}")
print(f"  - Asset ID: {training_data_asset_id}")
print(f"\n✅ Ready for model training!")

## Summary

This notebook has:
1. ✅ Generated synthetic input data matching the model's input schema
2. ✅ Saved the data as a CSV file
3. ✅ Uploaded the CSV to watsonx.ai project assets
4. ✅ Verified the upload

**Next Steps:**
- Use the uploaded data asset in the model training notebook